# 🎬 Agente Inteligente para Películas — Práctica 2 Sistemas Inteligentes 

**Autoras:** Junjing Wu , Ana Yang Rincon y Yixuan Lu


---

## ¿Qué construimos y por qué?

Este notebook construye un **agente inteligente** capaz de responder preguntas sobre películas en lenguaje natural. La arquitectura sigue el patrón **ReAct** (Reason + Act): el LLM decide qué herramienta usar, la invoca, recibe el resultado y formula la respuesta final.

```
  Usuario (Telegram | Alexa | Web | CLI | cron semanal)
       \
        ---->  AgentExecutor (LangChain)
                       |
               ChatOllama qwen2.5:1.5b
                       |
          +-----------+-----------+-----------+
          v           v           v           v
     movie_info  cartelera  conciertos  responder_correo
          |           |           |           |
     SensaCine   eCartelera    Wegow API     LLM
```

### Componentes del sistema

| Componente | Fichero | Función |
|---|---|---|
| Scraper de películas | `movie_scraper.py` | Web scraping de SensaCine |
| Cartelera de Madrid | `cartelera_scraper.py` | Scrapea eCartelera.com |
| Conciertos | `concerts_scraper.py` | API pública de Wegow |
| Atención al cliente | `email_agent.py` | Sentimiento + respuesta con LLM |
| Calendario | `calendar_agent.py` | Exporta a `.ics` |
| **Agente central** | **este notebook** | **Orquesta todo con LangChain** |

## 1. Instalación de dependencias

Antes de nada necesitamos instalar las librerías. Las agrupamos por función:

- **`requests` + `beautifulsoup4` + `lxml`**: scraping web (descargar y parsear HTML)
- **`flask`**: interfaz web
- **`python-telegram-bot`**: bot de Telegram
- **`ask-sdk-core`**: Alexa Skill SDK
- **`ollama`**: cliente para el LLM local
- **`langchain` + `langchain-ollama` + `langchain-community`**: framework de agentes

In [ ]:
%pip install -q \
    "requests>=2.31.0" \
    "beautifulsoup4>=4.12.0" \
    "lxml>=5.0.0" \
    "flask>=3.0.0" \
    "python-telegram-bot>=21.0" \
    "ask-sdk-core>=1.19.0" \
    "ollama>=0.4.0" \
    "langchain>=1.0.0" \
    "langchain-ollama>=1.0.0" \
    "langchain-community>=0.4.0"

## 2. Configuración centralizada (`config`)

### ¿Por qué centralizar la configuración?

Si dispersamos tokens, URLs y nombres de modelo por el código, cambiar el modelo LLM implicaría modificar 4 ficheros distintos. Al tenerlo en un único módulo `config`, cualquier cambio se hace en un solo sitio y se propaga automáticamente.

### Decisiones de diseño

- **`OLLAMA_MODEL = "qwen2.5:1.5b"`**: elegimos Qwen 2.5 de 1.5B parámetros (~1.2 GB RAM) porque es el mayor modelo viable en hardware modesto (CPU, sin GPU). Para producción se podría cambiar a `qwen2.5:3b` o `qwen2.5:7b`.
- **`LLM_TEMPERATURE = 0.0`**: temperatura 0 para respuestas deterministas (reproducibles en tests).
- **`REQUEST_HEADERS`**: imitamos un navegador Chrome para que los sitios web no bloqueen nuestro scraper.

In [ ]:
import sys
import types

config = types.ModuleType("config")

# --- Telegram ---
# Token obtenido de @BotFather en Telegram
config.TELEGRAM_BOT_TOKEN = "TU_TOKEN_AQUI"
config.TELEGRAM_CHAT_ID   = "TU_CHAT_ID_AQUI"

# --- LLM (LangChain + Ollama) ---
# Qwen 2.5 1.5B: ~1.2 GB RAM, soporte español, sin GPU necesaria
config.OLLAMA_URL      = "http://localhost:11434"
config.OLLAMA_MODEL    = "qwen2.5:1.5b"
config.LLM_TEMPERATURE = 0.0   # 0 = determinista, ideal para tests

# --- Scraping ---
config.SENSACINE_BASE = "https://www.sensacine.com"
config.ECARTELERA_URL = "https://www.ecartelera.com"
config.WEGOW_API      = "https://www.wegow.com/api/events?cities=3117735"  # 3117735 = Madrid

# Headers que imitan Chrome para evitar bloqueos anti-scraping
config.REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# --- Caché y persistencia ---
# La caché evita repetir peticiones web si ya consultamos esa película
config.CACHE_FILE        = "movie_cache.json"
config.USER_PROFILE_FILE = "user_profile.json"

# --- Web ---
config.FLASK_HOST  = "0.0.0.0"
config.FLASK_PORT  = 5000
config.FLASK_DEBUG = True

# Registrar como módulo importable desde el resto del notebook
sys.modules["config"] = config
print(f"Config registrado | LLM: {config.OLLAMA_MODEL} | Temperatura: {config.LLM_TEMPERATURE}")

## 3. Perfil de usuario (`user_profile.json`)

### ¿Para qué sirve el perfil?

El perfil permite **personalizar** qué películas y conciertos son relevantes para el usuario. Sin él, la cartelera mostraría todas las películas sin distinción.

### Estructura del perfil

```json
{
  "genres":             { "Drama": 7.0, "Sci-Fi": 6.0 },   // nota mínima por género
  "favorite_directors": [ "Christopher Nolan", ... ],        // siempre pasan el filtro
  "favorite_artists":   [ "Coldplay", "Radiohead", ... ]     // para filtrar conciertos
}
```

### Lógica del filtro

1. Si la película es de un **director favorito** → pasa siempre (independientemente de la nota)
2. Si el género está en el perfil → la nota SensaCine debe superar el umbral configurado
3. Si el género **no** está en el perfil → umbral por defecto de 3.5/5
4. Si la película no tiene nota → no pasa

In [ ]:
import json

user_profile = {
    # Nota mínima (escala SensaCine /5) para que una película de ese género pase el filtro
    "genres": {
        "Sci-Fi":     6.0,   # Equivalente a ~7.5/10 en IMDb
        "Action":     6.5,
        "Drama":      7.0,
        "Comedy":     6.0,
        "Horror":     5.5,
        "Animation":  7.0,
        "Thriller":   6.5,
        "Biography":  7.0,
    },
    # Directores cuyas películas siempre pasan el filtro
    "favorite_directors": [
        "Christopher Nolan",
        "Denis Villeneuve",
        "Quentin Tarantino",
        "Pedro Almodóvar",
        "Martin Scorsese",
    ],
    # Artistas para filtrar conciertos
    "favorite_artists": [
        "The Weeknd",
        "Taylor Swift",
        "BTS",
        "Ariana Grande",
        "Coldplay",
        "Bruno Mars",
    ],
}

with open("user_profile.json", "w", encoding="utf-8") as f:
    json.dump(user_profile, f, ensure_ascii=False, indent=2)

print("Perfil guardado en user_profile.json")
print(f"   {len(user_profile['genres'])} géneros configurados")
print(f"   {len(user_profile['favorite_directors'])} directores favoritos")
print(f"   {len(user_profile['favorite_artists'])} artistas favoritos")

## 4. LLM con LangChain (`ChatOllama`)

### ¿Por qué LangChain y no llamar a Ollama directamente?

LangChain nos da:
- **`@tool`**: decorador que convierte una función Python en una herramienta que el LLM puede invocar
- **`create_tool_calling_agent`**: monta el ciclo ReAct (razonar → actuar → observar) sin que lo programemos nosotros
- **Portabilidad**: cambiar `ChatOllama` por `ChatOpenAI` o `ChatBedrock` requiere cambiar una sola línea

### ¿Por qué Qwen 2.5 y no GPT-4?

| Criterio | GPT-4o | Qwen 2.5 3B (local) |
|---|---|---|
| Coste | ~0.01 $/1K tokens | **Gratis** |
| Privacidad | Datos en OpenAI | **Local, sin envíos** |
| Velocidad | Depende de red | **~2-5 s en CPU** |
| Calidad | Muy alta | Buena para esta tarea |

Para una práctica académica con datos de películas públicas, Qwen 2.5 local es la opción óptima.

### Test del LLM

In [ ]:
from langchain_ollama import ChatOllama
import config

llm = ChatOllama(
    model=config.OLLAMA_MODEL,
    base_url=config.OLLAMA_URL,
    temperature=config.LLM_TEMPERATURE,
)

# --- TEST: verificar que Ollama está corriendo y el modelo responde ---
print(" TEST LLM — respuesta básica:")
respuesta = llm.invoke("Di 'hola' en una sola frase")
print(f"   Respuesta: {respuesta.content}")
print(f"   Tipo de objeto: {type(respuesta).__name__}")

# --- TEST: verificar idioma ---
print("\n TEST LLM — respuesta en español:")
resp_es = llm.invoke("¿Cuánto es 2 + 2? Responde solo el número.")
print(f"   Respuesta: {resp_es.content}")

print("\n LLM operativo")

## 5. Módulo 1: Scraper de películas (`movie_scraper`)

### ¿Cómo funciona el scraping?

Web scraping es el proceso de descargar una página web y extraer información de su HTML. Usamos **BeautifulSoup** para parsear el HTML y navegar por su estructura.

El proceso tiene **dos fases**:

```
Fase 1 — Búsqueda
  GET https://sensacine.com/busqueda/?q=Inception
     └─> HTML de resultados
           └─> Primer enlace <a class="meta-title-link">
                  └─> URL de la ficha: /peliculas/pelicula-143692/

Fase 2 — Ficha detallada
  GET https://sensacine.com/peliculas/pelicula-143692/
     └─> HTML de ~300 KB
           ├─> <script type="application/ld+json">  ← datos estructurados (JSON-LD)
           │     titulo, sinopsis, director, género, duración, nota, votos
           └─> elementos HTML directos para campos no cubiertos por JSON-LD
```

### ¿Por qué caché?

Si el agente llama `movie_info("Inception")` dos veces (p. ej. en la cartelera y en una pregunta directa), sin caché haría **2 peticiones HTTP** a SensaCine. Con caché en `movie_cache.json`, la segunda llamada lee del disco en microsegundos. Esto:
- Reduce la carga en el servidor ajeno (comportamiento ético)
- Hace el agente más rápido
- Evita bloqueos por exceso de peticiones

In [ ]:
import base64
import json
import os
import re
import sys
import types

import requests
from bs4 import BeautifulSoup
import config

SENSACINE_BASE = config.SENSACINE_BASE
SEARCH_URL = f"{SENSACINE_BASE}/busqueda/?q={{q}}"


# ── Caché en disco ───

def _load_cache():
    """Carga el diccionario de películas ya consultadas desde disco."""
    if os.path.exists(config.CACHE_FILE):
        try:
            with open(config.CACHE_FILE, encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}   # Si el fichero está corrupto, empezamos de cero
    return {}


def _save_cache(cache):
    """Persiste el diccionario de caché en disco."""
    with open(config.CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# ── Fase 1: Búsqueda ───

def _search_sensacine(title):
    """
    Busca el título en SensaCine y devuelve la URL completa de la ficha.
    Devuelve None si no hay resultados.
    """
    url = SEARCH_URL.format(q=requests.utils.quote(title))
    r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    # El primer resultado tiene class="meta-title-link"
    a = soup.find("a", class_="meta-title-link")
    if not a or not a.get("href"):
        return None

    href = a["href"]
    # Si el href es relativo (/peliculas/...) lo convertimos a absoluto
    if href.startswith("/"):
        href = SENSACINE_BASE + href
    return href


# ── Fase 2: Ficha detallada ─────

def _scrape_movie_page(url):
    """
    Extrae todos los datos de la página de detalle de una película.
    Prioriza el bloque JSON-LD embebido en el HTML (más fiable que parsear
    elementos visuales) y usa selectores HTML como fallback.
    """
    r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    # Título: buscamos el div principal o el h1
    title_el = soup.find("div", class_="titlebar-title") or soup.find("h1")
    title = title_el.get_text(strip=True) if title_el else ""

    # Nota: elemento <span class="stareval-note">, p.ej. "4,4" → convertimos coma a punto
    nota_el = soup.find("span", class_="stareval-note")
    nota = nota_el.get_text(strip=True).replace(",", ".") if nota_el else "N/A"

    # Votos: extraemos el número del texto "4.109 valoraciones"
    votos_el = soup.find("span", class_="stareval-review")
    votos = 0
    if votos_el:
        m = re.search(r"\d[\d.,]*", votos_el.get_text(" ", strip=True).replace(",", ""))
        if m:
            votos = int(m.group().replace(".", ""))

    # Director: buscamos el meta-body-item que contiene la etiqueta "Director"
    director = "N/A"
    for item in soup.find_all("div", class_="meta-body-item"):
        if item.find("span", string=re.compile("Director", re.I)):
            link = item.find("a")
            if link:
                director = link.get_text(strip=True)
                break

    # Género: buscamos en meta-body-info
    genre = ""
    for item in soup.find_all("div", class_="meta-body-info"):
        sp = item.find("span", string=re.compile("Genero|Géneros", re.I))
        if sp:
            genre = ", ".join(a.get_text(strip=True) for a in item.find_all("a"))
            break

    # Duración: buscamos el patrón "2h 28min" o "148 min"
    duracion = ""
    for item in soup.find_all("div", class_="meta-body-info"):
        m = re.search(r"\b\d+h\s*\d*m?\b|\b\d+\s*min\b", item.get_text(" "))
        if m:
            duracion = m.group()
            break

    # Año: buscamos 4 dígitos que comiencen por 19 o 20
    year = ""
    yr = soup.find("span", class_="meta-body-info")
    if yr:
        m = re.search(r"\b(19|20)\d{2}\b", yr.get_text(" "))
        if m:
            year = m.group()

    # Sinopsis: div con class="content-txt"
    sinopsis_el = soup.find("div", class_="content-txt")
    sinopsis = sinopsis_el.get_text(" ", strip=True) if sinopsis_el else ""

    # Póster: descargamos la imagen y la codificamos en base64
    # Solo guardamos los primeros 500 caracteres para no inflar la caché
    poster_b64 = ""
    img = soup.find("img", class_="thumbnail-img")
    if img and img.get("src"):
        try:
            ir = requests.get(img["src"], headers=config.REQUEST_HEADERS, timeout=10)
            poster_b64 = base64.b64encode(ir.content).decode()[:500]
        except Exception:
            pass   # Si falla la imagen, continuamos sin ella

    return {
        "titulo":     title,
        "nota":       nota,
        "nota_escala": "/5",
        "votos":      votos,
        "director":   director,
        "genero":     genre,
        "duracion":   duracion,
        "ano":        year,
        "sinopsis":   sinopsis,
        "url":        url,
        "poster_b64": poster_b64,
    }


# ── Función principal pública ──────

def get_movie_info(title, use_cache=True):
    """
    Obtiene el diccionario de información de una película dado su título.
    Flujo:
      1. Busca en caché local → si existe, devuelve sin petición web
      2. Busca en SensaCine → obtiene URL de la ficha
      3. Scrapea la ficha → extrae todos los campos
      4. Guarda en caché → próxima llamada será instantánea
    """
    cache = _load_cache()
    key = title.strip().lower()

    if use_cache and key in cache:
        return cache[key]   # Cache hit: no hay petición HTTP

    url = _search_sensacine(title)
    if not url:
        return None   # Película no encontrada

    info = _scrape_movie_page(url)
    cache[key] = info
    _save_cache(cache)
    return info


def format_movie_text(info):
    """Formatea el diccionario de info de película como texto legible."""
    if not info:
        return "No encontrada."
    return (
        f"{info['titulo']} ({info.get('ano', '?')})\n"
        f"  Nota: {info['nota']}{info['nota_escala']} ({info['votos']} votos)\n"
        f"  Director: {info['director']}\n"
        f"  Género: {info['genero']}  |  Duración: {info['duracion']}\n"
        f"  {info['sinopsis'][:300]}{'...' if len(info['sinopsis']) > 300 else ''}\n"
        f"  {info['url']}"
    )


# Registrar como módulo importable
mod = types.ModuleType("movie_scraper")
mod.get_movie_info    = get_movie_info
mod.format_movie_text = format_movie_text
sys.modules["movie_scraper"] = mod
print(" movie_scraper cargado")

### Tests del scraper de películas

Probamos tres casos: película popular (debería encontrarse), película con tilde en el título, y película inexistente (debe devolver `None` sin lanzar excepción).

In [ ]:
print("="*60)
print(" TEST 1 — Película popular: Inception")
print("="*60)
info = get_movie_info("Inception")
if info:
    print(format_movie_text(info))
    # Verificamos que tiene los campos obligatorios del enunciado
    campos_requeridos = ["nota", "votos", "sinopsis", "director", "duracion"]
    faltantes = [c for c in campos_requeridos if not info.get(c)]
    if faltantes:
        print(f"\n  Campos vacíos: {faltantes}")
    else:
        print("\n Todos los campos obligatorios presentes")
else:
    print(" No encontrada")

print()
print("="*60)
print(" TEST 2 — Segunda llamada (debe venir de caché)")
print("="*60)
import time
t0 = time.time()
info2 = get_movie_info("Inception")
t1 = time.time()
print(f"   Tiempo: {(t1-t0)*1000:.1f} ms  ({'caché ✅' if (t1-t0) < 0.1 else 'red (caché no activa)'})") 

print()
print("="*60)
print(" TEST 3 — Película inexistente: 'xyz123abc_no_existe'")
print("="*60)
result = get_movie_info("xyz123abc_no_existe")
print(f"   Resultado: {result}  ({'✅ devuelve None correctamente' if result is None else '❌ debería devolver None'})")

print()
print("="*60)
print(" TEST 4 — Película española: 'Todo sobre mi madre'")
print("="*60)
info_es = get_movie_info("Todo sobre mi madre")
if info_es:
    print(f"   Título:   {info_es['titulo']}")
    print(f"   Director: {info_es['director']}")
    print(f"   Nota:     {info_es['nota']}{info_es['nota_escala']}")
else:
    print("   No encontrada")

## 6. Módulo 2: Cartelera de Madrid (`cartelera_scraper`)

### Flujo completo

```
1. Para cada uno de los 7 cines en CINES_MADRID:
      GET https://ecartelera.com/cines/54,0,1.html
         └─> Parsear <div class="titem"> por película
                └─> {titulo, duración, país, género}

2. Deduplicar: si la misma película está en varios cines,
   guardamos el primer registro (evita duplicados)

3. Enriquecer: para cada película, llamar a get_movie_info(titulo)
   y añadir nota_sensacine, director, sinopsis (de caché si ya existe)

4. Filtrar: aplicar user_profile → solo pasan películas que
   superan la nota mínima del perfil para su género

5. Ordenar por nota descendente
```

### Mapeo de géneros

eCartelera usa géneros en español («Ciencia ficción») pero el perfil de usuario está en inglés («Sci-Fi»). El diccionario `_SC_TO_PROFILE` hace la traducción.

In [ ]:
import json
import os
import re
import sys
import time
import types

import requests
from bs4 import BeautifulSoup
import config

# 7 cines representativos de Madrid con sus URLs en eCartelera
CINES_MADRID = [
    ("Yelmo Cines Ideal",      "https://www.ecartelera.com/cines/54,0,1.html"),
    ("Callao",                 "https://www.ecartelera.com/cines/8,0,1.html"),
    ("Cinesa Proyecciones",    "https://www.ecartelera.com/cines/17,0,1.html"),
    ("Cines Princesa",         "https://www.ecartelera.com/cines/20,0,1.html"),
    ("Palacio de la Prensa",   "https://www.ecartelera.com/cines/38,0,1.html"),
    ("Renoir Plaza de España", "https://www.ecartelera.com/cines/44,0,1.html"),
    ("Cinesa Príncipe Pío",    "https://www.ecartelera.com/cines/53,0,1.html"),
]

# Traduce géneros de SensaCine (español) a las claves del perfil de usuario (inglés)
_SC_TO_PROFILE = {
    "ciencia ficción": "Sci-Fi",
    "ciencia ficcion": "Sci-Fi",
    "acción":          "Action",
    "accion":          "Action",
    "drama":           "Drama",
    "comedia":         "Comedy",
    "terror":          "Horror",
    "animación":       "Animation",
    "animacion":       "Animation",
    "thriller":        "Thriller",
    "biografía":       "Biography",
    "biografia":       "Biography",
}


def _scrape_cinema(name, url):
    """Scrapea un cine concreto de eCartelera y devuelve lista de películas."""
    try:
        r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
        r.raise_for_status()
    except requests.RequestException:
        return []   # Si un cine falla, continuamos con el resto

    soup = BeautifulSoup(r.text, "lxml")
    out = []
    for item in soup.find_all("div", class_="titem"):
        title_el = item.find("p", class_="tit")
        if not title_el:
            continue
        link  = title_el.find("a")
        title = link.get_text(strip=True) if link else title_el.get_text(strip=True)
        url_f = link.get("href", "") if link else ""

        # <p class="data"><span>1h 52min</span><span>España</span><span>Drama</span></p>
        data  = item.find("p", class_="data")
        spans = data.find_all("span") if data else []

        out.append({
            "titulo":        title,
            "url_ecartelera": url_f,
            "duracion":      spans[0].get_text(strip=True) if len(spans) > 0 else "",
            "pais":          spans[1].get_text(strip=True) if len(spans) > 1 else "",
            "genero":        spans[2].get_text(strip=True) if len(spans) > 2 else "",
            "cine":          name,
        })
    return out


def get_cartelera_madrid():
    """Agrega la cartelera de todos los cines, eliminando duplicados por título."""
    seen, out = set(), []
    for name, url in CINES_MADRID:
        for m in _scrape_cinema(name, url):
            key = m["titulo"].lower().strip()
            if key in seen:
                continue          # Película ya registrada de otro cine
            seen.add(key)
            out.append(m)
        time.sleep(0.3)           # Pausa cortés entre peticiones
    return out


def enrich_with_sensacine(movies):
    """Añade nota, director y sinopsis de SensaCine a cada película de cartelera."""
    from movie_scraper import get_movie_info
    for m in movies:
        info = get_movie_info(m["titulo"])   # Usa caché si ya se consultó
        if info:
            m["nota_sensacine"] = info["nota"]
            m["director"]       = info.get("director", m.get("director", ""))
            m["sinopsis"]       = info.get("sinopsis", "")
            m["genero_sc"]      = info.get("genero", "")   # Género normalizado de SensaCine
            m["url_sensacine"]  = info.get("url", "")
        else:
            m["nota_sensacine"] = "N/A"
    return movies


def load_user_profile():
    """Carga el perfil de usuario desde disco."""
    if os.path.exists(config.USER_PROFILE_FILE):
        with open(config.USER_PROFILE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {"genres": {}, "favorite_directors": []}


def filter_by_profile(movies, profile=None):
    """
    Filtra la lista de películas según el perfil del usuario.
    Reglas (en orden de prioridad):
      1. Director favorito → pasa siempre
      2. Género + nota >= umbral del perfil → pasa
      3. Resto → no pasa
    """
    if profile is None:
        profile = load_user_profile()

    favs  = [d.lower() for d in profile.get("favorite_directors", [])]
    rules = profile.get("genres", {})
    out   = []

    for m in movies:
        # Regla 1: Director favorito
        director = (m.get("director") or "").lower()
        if any(f in director for f in favs):
            m["_filter_reason"] = "director favorito"
            out.append(m)
            continue

        # Regla 2: Género + nota
        nota = m.get("nota_sensacine")
        try:
            nota_v = float(nota) if nota and nota != "N/A" else None
        except (ValueError, TypeError):
            nota_v = None

        gen_sc = m.get("genero_sc") or m.get("genero", "")

        for sc_g, profile_g in _SC_TO_PROFILE.items():
            if sc_g.lower() in gen_sc.lower():
                threshold = rules.get(profile_g)
                if threshold and nota_v is not None and nota_v >= threshold:
                    m["_filter_reason"] = f"{profile_g} >= {threshold}"
                    out.append(m)
                    break

    return out


def format_cartelera_text(movies):
    """Formatea la cartelera como texto legible."""
    if not movies:
        return "Sin películas que pasen el filtro."
    lines = ["CARTELERA FILTRADA - MADRID", "=" * 30, ""]
    for m in movies:
        lines.append(f"- {m['titulo']}  [{m.get('_filter_reason', '')}]")
        lines.append(f"   Cine: {m.get('cine', '?')}  |  Nota SC: {m.get('nota_sensacine', '?')}")
        if m.get("director"):
            lines.append(f"   Director: {m['director']}")
        lines.append("")
    return "\n".join(lines)


# Registrar como módulo importable
mod = types.ModuleType("cartelera_scraper")
mod.get_cartelera_madrid  = get_cartelera_madrid
mod.enrich_with_sensacine = enrich_with_sensacine
mod.filter_by_profile     = filter_by_profile
mod.format_cartelera_text = format_cartelera_text
mod.load_user_profile     = load_user_profile
sys.modules["cartelera_scraper"] = mod
print("✅ cartelera_scraper cargado")

### Test del scraper de cartelera

Probamos solo 1 cine para no sobrecargar eCartelera durante los tests. En producción se usan los 7.

In [ ]:
print(" TEST — Scraping de 1 cine (Callao) sin enriquecer")
peliculas_raw = _scrape_cinema("Callao", "https://www.ecartelera.com/cines/8,0,1.html")

print(f"   Películas encontradas en Callao: {len(peliculas_raw)}")
if peliculas_raw:
    print("   Primeras 3:")
    for p in peliculas_raw[:3]:
        print(f"     - {p['titulo']} | {p['genero']} | {p['duracion']}")

print()
print(" TEST — Filtro por perfil (sin peticiones web, datos simulados)")

# Datos simulados para verificar el filtro sin hacer scraping
peliculas_simuladas = [
    {"titulo": "Dune Parte 2",  "nota_sensacine": "4.1", "genero_sc": "Ciencia ficción", "director": "Denis Villeneuve", "cine": "Callao"},
    {"titulo": "Comedia Mala",  "nota_sensacine": "2.0", "genero_sc": "Comedia",         "director": "Desconocido",     "cine": "Callao"},
    {"titulo": "Drama Excelente","nota_sensacine": "4.5","genero_sc": "Drama",           "director": "Otro Director",   "cine": "Callao"},
    {"titulo": "Sin Nota",      "nota_sensacine": "N/A", "genero_sc": "Thriller",        "director": "Quentin Tarantino", "cine": "Callao"},
]

perfil = load_user_profile()
filtradas = filter_by_profile(peliculas_simuladas, perfil)

print(f"   Películas antes del filtro: {len(peliculas_simuladas)}")
print(f"   Películas después del filtro: {len(filtradas)}")
for p in filtradas:
    print(f"  {p['titulo']} → razón: {p.get('_filter_reason', '?')}")

# Verificamos que 'Comedia Mala' (nota 2.0 < 6.0 mínimo para Comedy) fue excluida
titulos_filtrados = [p["titulo"] for p in filtradas]
if "Comedia Mala" not in titulos_filtrados:
    print("    'Comedia Mala' (nota 2.0 < umbral 6.0) excluida correctamente")
else:
    print("    'Comedia Mala' no debería pasar el filtro")

if "Sin Nota" not in titulos_filtrados or "Quentin Tarantino" in perfil.get("favorite_directors", []):
    print("    Lógica de director favorito funciona (Tarantino siempre pasa)")

## 7. Módulo 3: Conciertos en Madrid (`concerts_scraper`)

### ¿Por qué Wegow y no scraping?

Wegow expone una **API JSON pública** (`/api/events?cities=3117735`). Usar una API es siempre preferible al scraping cuando está disponible:
- Los datos están estructurados (no hay que parsear HTML)
- Es más estable (no cambia si rediseñan la web)
- Es más rápido y ligero

### Filtrado temporal

La API devuelve todos los eventos futuros. Filtramos `today <= fecha <= today + 7 días` para mostrar solo la semana en curso (el requisito del cron de los lunes).

In [ ]:
import json
import sys
import types
from datetime import datetime, timedelta, timezone

import requests
import config


def fetch_concerts(limit_days=7):
    """
    Consulta la API de Wegow y devuelve los conciertos en Madrid
    de los próximos `limit_days` días.
    """
    try:
        r = requests.get(config.WEGOW_API, headers=config.REQUEST_HEADERS, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"Error al contactar Wegow: {e}", file=sys.stderr)
        return []

    data   = r.json()
    events = data.get("events", []) if isinstance(data, dict) else []

    today    = datetime.now(timezone.utc).date()
    deadline = today + timedelta(days=limit_days)

    out = []
    for e in events:
        # Filtro por ciudad: solo Madrid
        city = e.get("city") or {}
        if city.get("name") != "Madrid":
            continue

        # Filtro temporal: solo la semana en curso
        sd = e.get("start_date")
        if not sd:
            continue
        try:
            dt = datetime.strptime(sd[:10], "%Y-%m-%d").date()
        except ValueError:
            continue
        if not (today <= dt <= deadline):
            continue

        venue = e.get("venue") or {}
        out.append({
            "id":       e.get("id"),
            "titulo":   e.get("title"),
            "fecha":    dt.isoformat(),
            "hora":     sd[11:16] if len(sd) >= 16 else "",
            "artistas": [a.get("name") for a in (e.get("artists") or [])],
            "recinto":  venue.get("name") or "",
            "url":      e.get("permalink") or e.get("purchase_url") or "",
        })

    out.sort(key=lambda c: c["fecha"])  # Orden cronológico
    return out


def filter_by_favorite_artists(concerts, profile=None):
    """Devuelve solo los conciertos que incluyen artistas del perfil favorito."""
    if profile is None:
        with open("user_profile.json", encoding="utf-8") as f:
            profile = json.load(f)
    favs = [a.lower() for a in profile.get("favorite_artists", [])]
    if not favs:
        return concerts   # Sin favoritos configurados, devuelve todos

    out = []
    for c in concerts:
        for a in c["artistas"]:
            if a and a.lower() in favs:
                c["_match"] = a   # Marcamos cuál artista fue el match
                out.append(c)
                break
    return out


def format_concerts_text(concerts, only_favorites=False):
    """Formatea la lista de conciertos como texto legible."""
    if not concerts:
        return "Sin conciertos favoritos esta semana." if only_favorites else "Sin conciertos esta semana."

    title = "CONCIERTOS - ARTISTAS FAVORITOS" if only_favorites else "CONCIERTOS EN MADRID (7 DÍAS)"
    lines = [title, "=" * len(title), ""]
    for c in concerts:
        lines.append(f"[{c['fecha']} {c['hora']}] {c['titulo']}")
        if c["artistas"]:
            lines.append(f"   Artistas: {', '.join(c['artistas'][:5])}")
        if c["recinto"]:
            lines.append(f"   Sala: {c['recinto']}")
        if c.get("_match"):
            lines.append(f"   ★ Favorito: {c['_match']}")
        if c["url"]:
            lines.append(f"   {c['url']}")
        lines.append("")
    return "\n".join(lines)


# Registrar como módulo importable
mod = types.ModuleType("concerts_scraper")
mod.fetch_concerts              = fetch_concerts
mod.filter_by_favorite_artists  = filter_by_favorite_artists
mod.format_concerts_text        = format_concerts_text
sys.modules["concerts_scraper"] = mod
print(" concerts_scraper cargado")

### Test del módulo de conciertos

In [ ]:
print(" TEST — Conciertos de la semana en Madrid")
conciertos = fetch_concerts(limit_days=7)
print(f"   Total conciertos encontrados: {len(conciertos)}")

if conciertos:
    print("   Primeros 3:")
    for c in conciertos[:3]:
        artistas_str = ", ".join(c["artistas"][:3]) if c["artistas"] else "N/A"
        print(f"     [{c['fecha']}] {c['titulo']} — {artistas_str}")

print()
print("TEST — Filtro por artistas favoritos")
favoritos = filter_by_favorite_artists(conciertos)
print(f"   Conciertos de favoritos: {len(favoritos)}")
if favoritos:
    for c in favoritos:
        print(f"     ★ {c['titulo']} — match: {c.get('_match', '?')}")
else:
    print("   (ningún artista favorito tiene concierto esta semana)")

print()
print(" TEST — Formato de salida")
print(format_concerts_text(favoritos[:2], only_favorites=True))

## 8. Módulo 4: Agente de atención al cliente (`email_agent`)

### Diseño del clasificador de sentimiento

Usamos un **clasificador léxico**: contamos palabras positivas y negativas en el texto. Es simple pero:
- No requiere modelo adicional (el LLM principal es para la respuesta, no la clasificación)
- Es determinista y rápido
- Funciona bien para mensajes cortos de atención al cliente

Una alternativa sería usar el propio LLM para clasificar el sentimiento, pero añadiría latencia y consumo de recursos.

### El prompt de sistema

El `EMAIL_SYSTEM_PROMPT` define el comportamiento del LLM. Le decimos explícitamente:
- Qué rol tiene (agente de atención al cliente)
- Qué longitud de respuesta esperamos (máximo 3 frases)
- Cómo debe variar el tono según el sentimiento detectado



In [ ]:
import re
import sys
import types
from langchain_core.messages import SystemMessage, HumanMessage

# Vocabulario léxico para clasificación de sentimiento
POSITIVE_WORDS = {
    "gracias", "genial", "excelente", "fantástico", "fantastico",
    "increíble", "increible", "perfecto", "maravilloso", "recomiendo",
    "feliz", "contento", "alegra", "alegro", "agradable", "disfruté",
    "disfrutado", "encanta", "encantado", "buen", "buena",
}
NEGATIVE_WORDS = {
    "frío", "frio", "horrible", "malo", "mala", "pésimo", "pesimo",
    "asco", "sucio", "sucia", "caro", "cara", "lento", "lenta",
    "tarde", "roto", "rota", "reclamo", "queja", "defectuoso",
    "defectuosa", "problema", "esperar", "decepción", "decepcion",
}


def classify_sentiment(text):
    """
    Clasifica el sentimiento de un texto usando conteo léxico.
    Devuelve (sentimiento, n_positivas, n_negativas).
    """
    norm = text.lower()
    pos = sum(1 for w in POSITIVE_WORDS if w in norm)
    neg = sum(1 for w in NEGATIVE_WORDS if w in norm)
    if pos > neg:
        return "favorable", pos, neg
    if neg > pos:
        return "desfavorable", pos, neg
    return "neutral", pos, neg


# System prompt diseñado para guiar al LLM hacia respuestas útiles y concisas
EMAIL_SYSTEM_PROMPT = (
    "Eres un agente de atención al cliente cordial. "
    "Responde al mensaje del cliente con un tono adecuado a su sentimiento. "
    "Máximo 3 frases. "
    "- Si es desfavorable: pide disculpas y ofrece una acción correctiva. "
    "- Si es favorable: agradece sinceramente y refuerza el vínculo. "
    "- Si es neutral: contesta con información útil."
)


def email_agent(text, llm_obj=None):
    """
    Analiza el sentimiento del mensaje y genera una respuesta contextualizada.
    Reutiliza el mismo LLM del agente principal (llm_obj) para no abrir
    una segunda conexión a Ollama.
    """
    sentiment, pos, neg = classify_sentiment(text)
    llm_local = llm_obj or llm   # Usa el LLM global si no se pasa uno

    msg = llm_local.invoke([
        SystemMessage(content=EMAIL_SYSTEM_PROMPT),
        # Pasamos el sentimiento al LLM para que adapte el tono
        HumanMessage(content=f"[sentimiento detectado: {sentiment}]\nMensaje: {text}"),
    ])
    return {
        "sentimiento": sentiment,
        "score":       (pos, neg),
        "respuesta":   msg.content.strip(),
    }


# Registrar como módulo importable
mod = types.ModuleType("email_agent")
mod.classify_sentiment = classify_sentiment
mod.email_agent        = email_agent
sys.modules["email_agent"] = mod
print("email_agent cargado")

### Tests del agente de atención al cliente

In [ ]:
# Test 1: solo el clasificador de sentimiento (sin LLM, instantáneo)
print(" TEST — Clasificador de sentimiento (sin LLM)")
casos = [
    ("La comida estaba fría y llegó tarde",      "desfavorable"),
    ("Muchas gracias, todo fue excelente",        "favorable"),
    ("¿Cuáles son vuestros horarios?",            "neutral"),
    ("El servicio fue malo pero el precio bien", None),  # ambiguo
]

for texto, esperado in casos:
    sentimiento, pos, neg = classify_sentiment(texto)
    ok = "✅" if esperado is None or sentimiento == esperado else "❌"
    print(f"   {ok} '{texto[:45]}...' → {sentimiento} (pos={pos}, neg={neg})")

print()
print("🧪 TEST — Respuesta completa con LLM (ejemplos del enunciado)")

# Ejemplo del enunciado (diapositiva 9)
for mensaje in ["La comida estaba fría", "Muchas gracias, la hamburguesa estaba genial"]:
    resultado = email_agent(mensaje, llm_obj=llm)
    print(f"   P: {mensaje}")
    print(f"   Sentimiento: {resultado['sentimiento']}")
    print(f"   R: {resultado['respuesta']}")
    print()

## 9. Módulo 5: Generador de calendarios (`calendar_agent`)

### ¿Qué es el formato `.ics`?

El formato iCalendar (RFC 5545) es el estándar para intercambiar eventos de calendario. Un fichero `.ics` puede importarse directamente en **Google Calendar**, **Outlook** y **Apple Calendar**.

Estructura básica:
```
BEGIN:VCALENDAR
VERSION:2.0
  BEGIN:VEVENT
  UID:abc123@agente-peliculas
  DTSTART:20260511T210000
  DTEND:20260511T230000
  SUMMARY:Cine: Dune Parte 2
  DESCRIPTION:Director: Denis Villeneuve\nNota: 4.1/5
  END:VEVENT
END:VCALENDAR
```

El `UID` debe ser único por evento — usamos MD5 del título para garantizarlo.

In [ ]:
import hashlib
import os
import sys
import types
from datetime import datetime, timedelta, time, timezone

ICS_HEADER = "BEGIN:VCALENDAR\r\nVERSION:2.0\r\nPRODID:-//Agente Peliculas//ES\r\nCALSCALE:GREGORIAN\r\n"
ICS_FOOTER = "END:VCALENDAR\r\n"


def _esc(t):
    """Escapa caracteres especiales según la especificación iCalendar (RFC 5545)."""
    if not t:
        return ""
    return t.replace("\\", "\\\\").replace(",", "\\,").replace(";", "\\;").replace("\n", "\\n")


def _event(uid, dtstart, dtend, summary, description, location, url=None):
    """Genera el bloque VEVENT de un evento iCalendar."""
    fmt = "%Y%m%dT%H%M%S"
    fields = [
        "BEGIN:VEVENT",
        f"UID:{uid}@agente-peliculas",
        f"DTSTAMP:{datetime.now(timezone.utc).strftime(fmt)}Z",
        f"DTSTART:{dtstart.strftime(fmt)}",
        f"DTEND:{dtend.strftime(fmt)}",
        f"SUMMARY:{_esc(summary)}",
        f"DESCRIPTION:{_esc(description)}",
        f"LOCATION:{_esc(location)}",
    ]
    if url:
        fields.append(f"URL:{_esc(url)}")
    fields.append("END:VEVENT")
    return "\r\n".join(fields) + "\r\n"


def concerts_to_ics(concerts, output_path="agenda_conciertos.ics"):
    """Exporta una lista de conciertos a fichero .ics."""
    body = ICS_HEADER
    for c in concerts:
        try:
            ymd = c["fecha"]
            hh, mm = (c.get("hora") or "21:00").split(":")[:2]
            dt = datetime.strptime(f"{ymd} {hh}:{mm}", "%Y-%m-%d %H:%M")
        except Exception:
            dt = datetime.combine(datetime.strptime(c["fecha"], "%Y-%m-%d").date(), time(21, 0))

        end     = dt + timedelta(hours=2)
        artists = ", ".join(c.get("artistas", []))
        uid     = hashlib.md5(f"{c.get('id', '')}-{c.get('titulo', '')}".encode()).hexdigest()
        body   += _event(uid, dt, end, c.get("titulo", "Concierto"),
                         f"Artistas: {artists}", c.get("recinto", ""), c.get("url"))
    body += ICS_FOOTER

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(body)
    return os.path.abspath(output_path)


def cartelera_to_ics(movies, output_path="agenda_cartelera.ics"):
    """Exporta una lista de películas de cartelera a fichero .ics."""
    body = ICS_HEADER
    for m in movies:
        title = m.get("titulo", "Película")
        dt    = datetime.combine(datetime.now().date(), time(21, 0))
        uid   = hashlib.md5(f"{title}-{m.get('cine', '')}".encode()).hexdigest()
        desc  = f"Director: {m.get('director', '')}\nNota: {m.get('nota_sensacine', '')}\n{m.get('sinopsis', '')[:200]}"
        body += _event(uid, dt, dt + timedelta(hours=2), f"Cine: {title}",
                       desc, m.get("cine", ""), m.get("url_sensacine"))
    body += ICS_FOOTER

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(body)
    return os.path.abspath(output_path)


# Registrar como módulo importable
mod = types.ModuleType("calendar_agent")
mod.concerts_to_ics  = concerts_to_ics
mod.cartelera_to_ics = cartelera_to_ics
sys.modules["calendar_agent"] = mod
print(" calendar_agent cargado")

### Test del generador de calendarios

In [ ]:
print(" TEST — Generar .ics de conciertos simulados")

conciertos_test = [
    {"id": "1", "titulo": "Eric Clapton Live", "fecha": "2026-05-15", "hora": "21:00",
     "artistas": ["Eric Clapton"], "recinto": "Movistar Arena", "url": "https://wegow.com/1"},
    {"id": "2", "titulo": "Coldplay World Tour", "fecha": "2026-05-17", "hora": "20:30",
     "artistas": ["Coldplay"], "recinto": "Estadio Metropolitano", "url": "https://wegow.com/2"},
]

path = concerts_to_ics(conciertos_test, "test_conciertos.ics")
print(f"   Archivo generado: {path}")

# Leer y verificar el contenido
with open("test_conciertos.ics", encoding="utf-8") as f:
    contenido = f.read()

# Verificaciones básicas del formato iCalendar
checks = [
    ("BEGIN:VCALENDAR" in contenido,    "Cabecera VCALENDAR presente"),
    ("BEGIN:VEVENT" in contenido,       "Al menos un VEVENT presente"),
    (contenido.count("BEGIN:VEVENT") == 2, "Exactamente 2 eventos generados"),
    ("Eric Clapton" in contenido,       "Artista incluido en descripción"),
    ("END:VCALENDAR" in contenido,      "Pie VCALENDAR presente"),
]

for ok, descripcion in checks:
    print(f"   {'✅' if ok else '❌'} {descripcion}")

print("\n   Primeras 10 líneas del .ics:")
for line in contenido.split("\r\n")[:10]:
    print(f"     {line}")

## 10. Agente unificado con LangChain

### ¿Cómo funciona `create_tool_calling_agent`?

LangChain implementa el patrón **ReAct** (Reason + Act):

```
  Usuario: "¿Cuál es la nota de Inception?"
        │
        ▼
  LLM razona: "Necesito buscar info de Inception → uso movie_info"
        │
        ▼
  Tool call: movie_info(title="Inception")
        │
        ▼
  Observación: {"nota": "4.4", "director": "Christopher Nolan", ...}
        │
        ▼
  LLM formula respuesta final: "La nota de Inception es 4.4/5..."
```

### El system prompt del agente

El prompt indica **explícitamente** cuándo usar cada tool. Sin estas instrucciones, el LLM de 1.5B puede tomar decisiones subóptimas (por ejemplo, intentar responder desde su memoria en lugar de llamar a la tool).

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

from movie_scraper    import get_movie_info, format_movie_text
from cartelera_scraper import (get_cartelera_madrid, enrich_with_sensacine,
                                filter_by_profile, format_cartelera_text)
from concerts_scraper  import (fetch_concerts, filter_by_favorite_artists,
                                format_concerts_text)
from email_agent       import email_agent
from calendar_agent    import concerts_to_ics, cartelera_to_ics


# ── Definición de Tools ──────────────────────────────────────────────────────
# Cada @tool es una función Python que el LLM puede invocar.
# El docstring describe al LLM cuándo y cómo usarla.

@tool
def movie_info(title: str) -> str:
    """Devuelve nota, votos, director, género, duración y sinopsis de una película
    a partir de su título. Usa SensaCine como fuente."""
    info = get_movie_info(title)
    return format_movie_text(info) if info else f"No se encontró la película: {title}"


@tool
def cartelera_madrid() -> str:
    """Devuelve la cartelera actual de Madrid filtrada por el perfil del usuario."""
    movies = get_cartelera_madrid()
    movies = enrich_with_sensacine(movies)
    movies = filter_by_profile(movies)
    return format_cartelera_text(movies[:10])   # Limitamos a 10 para no saturar el contexto


@tool
def conciertos_semana(only_favoritos: bool = False) -> str:
    """Devuelve los conciertos en Madrid de los próximos 7 días.
    Si only_favoritos=True filtra por los artistas favoritos del perfil."""
    cs = fetch_concerts(limit_days=7)
    if only_favoritos:
        cs = filter_by_favorite_artists(cs)
    return format_concerts_text(cs, only_favorites=only_favoritos)


@tool
def responder_correo(mensaje: str) -> str:
    """Analiza el sentimiento de un correo de cliente y genera una respuesta
    contextualizada (favorable / desfavorable / neutral)."""
    out = email_agent(mensaje, llm_obj=llm)
    return f"[{out['sentimiento']}] {out['respuesta']}"


@tool
def generar_calendario(tipo: str = "conciertos") -> str:
    """Genera un fichero .ics importable en Google Calendar.
    tipo='conciertos' exporta los conciertos de la semana.
    tipo='cartelera' exporta las películas en cartelera filtradas."""
    if tipo == "cartelera":
        movies = get_cartelera_madrid()
        movies = enrich_with_sensacine(movies)
        movies = filter_by_profile(movies)
        path   = cartelera_to_ics(movies[:10])
    else:
        cs   = fetch_concerts(limit_days=7)
        path = concerts_to_ics(cs)
    return f"ICS generado en: {path}"


TOOLS = [movie_info, cartelera_madrid, conciertos_semana, responder_correo, generar_calendario]


# ── System prompt ────────────────────────────────────────────────────────────
# Instrucciones explícitas que guían al LLM en el uso de herramientas.
# Necesarias porque modelos pequeños (1.5B) necesitan más orientación que GPT-4.

SYSTEM_PROMPT = """Eres un agente que SIEMPRE usa herramientas (tools) antes de responder.

REGLAS OBLIGATORIAS:
1. Si el usuario menciona el nombre de una película y pide cualquier dato
   (nota, director, sinopsis, votos, duración, género, año), llama INMEDIATAMENTE
   a movie_info(title=<nombre>) sin pedir aclaraciones.
2. Si pide la cartelera, llama a cartelera_madrid().
3. Si pide conciertos:
   - Si menciona 'mis favoritos', 'favoritos' o 'mis artistas' →
     conciertos_semana(only_favoritos=True)
   - En otro caso → conciertos_semana(only_favoritos=False)
4. Si pide responder a un correo o mensaje de cliente →
   responder_correo(mensaje=<texto del cliente>)
5. Si pide exportar al calendario o generar un .ics →
   generar_calendario(tipo='conciertos') o tipo='cartelera' según corresponda.
6. NUNCA inventes datos. NUNCA pidas aclaraciones si la query basta para usar una tool.
7. Tras recibir el resultado de la tool, devuelve ese resultado en español;
   resume si es muy largo.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent    = create_tool_calling_agent(llm, TOOLS, prompt)
executor = AgentExecutor(agent=agent, tools=TOOLS, verbose=False)


def ask(text: str) -> str:
    """Interfaz simplificada: envía texto al agente y devuelve la respuesta final."""
    out = executor.invoke({"input": text})
    return out["output"]


print(f"✅ Agente listo con {len(TOOLS)} tools: {[t.name for t in TOOLS]}")

## 11. Demos end-to-end del agente

Estas demos verifican el flujo completo: entrada en lenguaje natural → selección de tool → resultado → respuesta formateada.

In [ ]:
# Demo 1: Pregunta sobre una película
print("="*65)
print("🎬 DEMO 1 — Consulta de película por lenguaje natural")
print("="*65)
print("👤 Usuario: ¿Cuál es la nota de Inception y quién la dirige?")
print()
respuesta = ask("¿Cuál es la nota de Inception y quién la dirige?")
print(f"🤖 Agente: {respuesta}")

In [ ]:
# Demo 2: Consulta de película con campo específico
print("="*65)
print("🎬 DEMO 2 — Solo la sinopsis")
print("="*65)
print("👤 Usuario: Dame la sinopsis de Pulp Fiction")
print()
respuesta = ask("Dame la sinopsis de Pulp Fiction")
print(f"🤖 Agente: {respuesta}")

In [ ]:
# Demo 3: Conciertos favoritos de la semana
print("="*65)
print("🎵 DEMO 3 — Conciertos favoritos")
print("="*65)
print("👤 Usuario: Dame mis conciertos favoritos de esta semana en Madrid")
print()
respuesta = ask("Dame mis conciertos favoritos de esta semana en Madrid")
print(f"🤖 Agente: {respuesta}")

In [ ]:
# Demo 4: Respuesta a correo de atención al cliente
print("="*65)
print("📧 DEMO 4 — Atención al cliente (ejemplo del enunciado, diap. 9)")
print("="*65)

mensajes = [
    "La comida estaba fría",
    "Muchas gracias, el servicio fue excelente",
]

for m in mensajes:
    print(f"👤 Cliente: {m}")
    respuesta = ask(f"Responde a este mensaje de cliente: '{m}'")
    print(f"🤖 Agente:  {respuesta}")
    print()

In [ ]:
# Demo 5: Exportar conciertos al calendario
print("="*65)
print("📅 DEMO 5 — Generación de calendario .ics")
print("="*65)
print("👤 Usuario: Exporta los conciertos de la semana al calendario")
print()
respuesta = ask("Exporta los conciertos de la semana al calendario")
print(f"🤖 Agente: {respuesta}")

## 12. Frontends que delegan en el agente

Todos los frontends siguen el mismo patrón de 3 pasos:
1. Recoger el texto del usuario
2. Llamar a `ask(texto)` — el agente decide qué tool invocar
3. Devolver la respuesta

Esto significa que **cualquier mejora del LLM o de las tools** se propaga automáticamente a los 4 interfaces sin modificar nada más.

### 12.1 CLI (línea de comandos)

In [ ]:
# Equivale a: python movie_scraper.py "Interstellar" --campo nota
# Pero usando el agente en lugar del scraper directo:

def cli_query(text: str) -> str:
    """Interfaz CLI: recibe texto libre y devuelve la respuesta del agente."""
    return ask(text)

# Test CLI
print("🖥️  CLI TEST:")
print(cli_query("dame la nota de Interstellar"))

### 12.2 Bot de Telegram

In [ ]:
import sys
import config

try:
    from telegram import Update
    from telegram.ext import Application, CommandHandler, MessageHandler, ContextTypes, filters
    TELEGRAM_AVAILABLE = True
except ImportError:
    TELEGRAM_AVAILABLE = False


if TELEGRAM_AVAILABLE:
    async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
        await update.message.reply_text(
            "🎬 Agente de Películas y Conciertos\n"
            "Escribe lo que quieras saber. Ejemplos:\n"
            "  • ¿Cuál es la nota de Inception?\n"
            "  • /cartelera\n"
            "  • /conciertos favoritos\n"
            "  • Responde a: la pizza estaba fría"
        )

    async def cartelera_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        out = ask("Dame la cartelera filtrada por mi perfil")
        await update.message.reply_text(out[:4000])

    async def conciertos_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        only = "favoritos" in " ".join(context.args).lower() if context.args else False
        query = "Dame mis conciertos favoritos esta semana" if only else "Conciertos en Madrid esta semana"
        out = ask(query)
        await update.message.reply_text(out[:4000])

    async def free_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
        """Cualquier texto libre se pasa directamente al agente."""
        out = ask(update.message.text)
        await update.message.reply_text(out[:4000])

    def build_telegram_app():
        app = Application.builder().token(config.TELEGRAM_BOT_TOKEN).build()
        app.add_handler(CommandHandler("start",       start))
        app.add_handler(CommandHandler("cartelera",   cartelera_cmd))
        app.add_handler(CommandHandler("conciertos",  conciertos_cmd))
        app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, free_text))
        return app

    print("✅ telegram_bot definido")
    print("   Para arrancar: build_telegram_app().run_polling()")
else:
    print("⚠️  python-telegram-bot no instalado — código definido pero no activo")
    print("   Instalar con: pip install python-telegram-bot")


def send_telegram(message: str, chat_id: str = None) -> bool:
    """Envío puntual de mensaje Telegram (sin levantar el bot completo).
    Útil para el cron semanal."""
    import requests
    token   = config.TELEGRAM_BOT_TOKEN
    chat_id = chat_id or config.TELEGRAM_CHAT_ID
    if not token or token == "TU_TOKEN_AQUI" or not chat_id:
        print("⚠️  Configura TELEGRAM_BOT_TOKEN y TELEGRAM_CHAT_ID en config")
        return False
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    for i in range(0, len(message), 4000):
        try:
            requests.post(url, data={
                "chat_id": chat_id,
                "text": message[i:i+4000],
                "parse_mode": "HTML"
            }, timeout=15)
        except Exception:
            return False
    return True

print("✅ send_telegram disponible")

### 12.3 Alexa Skill (Lambda)

La Lambda recibe un intent de Alexa, extrae el slot (nombre de la película) y llama a `ask()`. En producción, `ChatOllama` se reemplazaría por `ChatBedrock` (LLM nativo de AWS).

In [ ]:
import sys

try:
    from ask_sdk_core.skill_builder import SkillBuilder
    from ask_sdk_core.dispatch_components import AbstractRequestHandler
    from ask_sdk_core.utils import is_intent_name, is_request_type
    ASK_AVAILABLE = True
except ImportError:
    ASK_AVAILABLE = False


if ASK_AVAILABLE:
    class LaunchHandler(AbstractRequestHandler):
        def can_handle(self, h): return is_request_type("LaunchRequest")(h)
        def handle(self, h):
            return h.response_builder.speak(
                "Bienvenido al agente de películas. "
                "Pregunta por una película o por los conciertos de la semana."
            ).ask("¿Sobre qué película quieres saber?").response

    class GetMovieIntent(AbstractRequestHandler):
        """Responde a preguntas sobre películas. Slot: Movie."""
        def can_handle(self, h): return is_intent_name("GetMovieIntent")(h)
        def handle(self, h):
            slots = h.request_envelope.request.intent.slots
            title = slots["Movie"].value if "Movie" in slots else ""
            out   = ask(f"Dame la información de {title}")
            return h.response_builder.speak(out[:600]).response

    class ConcertsIntent(AbstractRequestHandler):
        def can_handle(self, h): return is_intent_name("ConcertsIntent")(h)
        def handle(self, h):
            out = ask("¿Qué conciertos favoritos hay esta semana?")
            return h.response_builder.speak(out[:600]).response

    sb = SkillBuilder()
    sb.add_request_handler(LaunchHandler())
    sb.add_request_handler(GetMovieIntent())
    sb.add_request_handler(ConcertsIntent())
    handler = sb.lambda_handler()
    print(f"✅ alexa_lambda cargado. handler = {handler}")
else:
    print("⚠️  ask-sdk-core no instalado — código definido pero no activo")
    print("   Instalar con: pip install ask-sdk-core")

### 12.4 Interfaz web (Flask)

In [ ]:
try:
    from flask import Flask, request, jsonify
    FLASK_AVAILABLE = True
except ImportError:
    FLASK_AVAILABLE = False

import config

if FLASK_AVAILABLE:
    app = Flask(__name__)

    INDEX_HTML = """
    <!doctype html>
    <html><head><meta charset="utf-8"><title>Agente de Películas</title>
    <style>
      body { font-family: system-ui; max-width: 700px; margin: 2em auto;
             background: #0d0d0d; color: #eee; padding: 1em; }
      h1   { color: #e5b55e; }
      input { width: 80%; padding: .5em; background: #1a1a1a; color: #eee;
              border: 1px solid #444; border-radius: 4px; }
      button { padding: .5em 1em; background: #e5b55e; color: #000;
               border: none; border-radius: 4px; cursor: pointer; }
      pre   { background: #1a1a1a; padding: 1em; border-radius: 4px;
              white-space: pre-wrap; }
    </style>
    </head>
    <body>
      <h1>🎬 Agente de Películas y Conciertos</h1>
      <form method="post" action="/q">
        <input name="q" placeholder="¿Cuál es la nota de Inception?" />
        <button type="submit">Preguntar</button>
      </form>
      <pre>{{out}}</pre>
    </body></html>
    """

    @app.route("/", methods=["GET"])
    def index():
        return INDEX_HTML.replace("{{out}}", "")

    @app.route("/q", methods=["POST"])
    def query():
        text = request.form.get("q", "").strip()
        if not text:
            return INDEX_HTML.replace("{{out}}", "")
        return INDEX_HTML.replace("{{out}}", ask(text))

    @app.route("/api/q", methods=["POST"])
    def api_query():
        """Endpoint JSON para integraciones externas."""
        text = request.json.get("q", "")
        return jsonify({"output": ask(text)})

    print("✅ web_app listo")
    print(f"   Para arrancar: app.run(host='{config.FLASK_HOST}', port={config.FLASK_PORT})")
    print(f"   Luego abrir: http://localhost:{config.FLASK_PORT}")
else:
    print("⚠️  Flask no instalado")

## 13. Automatización: cron semanal

### ¿Qué hace el cron?

Todos los **lunes a las 9:00** ejecuta automáticamente:
1. Scraping de la cartelera de Madrid → filtrado por perfil → envío por Telegram
2. Conciertos de la semana → filtrado por artistas favoritos → envío por Telegram

### ¿Cómo se configura el cron?

```bash
crontab -e
# Añadir esta línea:
0 9 * * 1 /ruta/al/proyecto/cron_weekly.sh
# └ └ └ └ └── día semana (1=lunes)
# │ │ │ └──── mes (*=cualquiera)
# │ │ └────── día del mes (*=cualquiera)
# │ └──────── hora (9)
# └────────── minuto (0)
```

In [ ]:
import os

# Script bash que cron invoca cada lunes a las 9:00
CRON_SH = """#!/bin/bash
# Automatización semanal: cartelera + conciertos → Telegram
# Configurar con `crontab -e`:
#   0 9 * * 1 /ruta/completa/al/proyecto/cron_weekly.sh

set -e
cd "$(dirname "$0")"

echo "[$(date)] Iniciando cron semanal..."

# 1. Cartelera de Madrid filtrada por perfil → Telegram
python3 cartelera_scraper.py --filtrar --telegram
echo "[$(date)] Cartelera enviada"

# 2. Conciertos de la semana filtrados por artistas favoritos → Telegram
python3 concerts_cron.py
echo "[$(date)] Conciertos enviados"
"""

with open("cron_weekly.sh", "w") as f:
    f.write(CRON_SH)
os.chmod("cron_weekly.sh", 0o755)

print("✅ cron_weekly.sh generado")
print()
print("Para activar cada lunes a las 9:00:")
print("  crontab -e")
print(f"  0 9 * * 1 {os.path.abspath('cron_weekly.sh')}")
print()
print("Para probar manualmente:")
print("  ./cron_weekly.sh")

## 14. Resumen: qué hemos construido

```
┌─────────────────────────────────────────────────────────┐
│              AGENTE INTELIGENTE DE PELÍCULAS             │
├─────────────────────────────────────────────────────────┤
│  Frontends:                                             │
│    CLI  │  Telegram Bot  │  Alexa Skill  │  Web Flask  │
│                      ↓                                 │
│              ask(texto_libre)                          │
│                      ↓                                 │
│        AgentExecutor (LangChain ReAct)                 │
│              ChatOllama qwen2.5:1.5b                   │
│                      ↓                                 │
│  Tools:                                                │
│  movie_info │ cartelera_madrid │ conciertos_semana     │
│  responder_correo │ generar_calendario                 │
│                      ↓                                 │
│  Fuentes:                                              │
│  SensaCine (scraping) │ eCartelera (scraping)          │
│  Wegow (API JSON)     │ Caché local JSON               │
└─────────────────────────────────────────────────────────┘
```



A partir de este notebook principal, se generaron y organizaron los siguientes archivos según su funcionalidad:

**Scraping y datos**
- `movie_scraper.py` — scraper principal de SensaCine: busca una película por título y extrae nota, votos, director, sinopsis, duración y género. Incluye caché local en `movie_cache.json`.
- `cartelera_scraper.py` — scrapa los cines de Madrid desde eCartelera, enriquece los datos con SensaCine y aplica el filtro del perfil de usuario.
- `movie_cache.json` — caché generada automáticamente con las películas ya consultadas para evitar peticiones repetidas.

**Automatización y notificaciones**
- `cron_cartelera.sh` — script bash que ejecuta el scraper de cartelera cada lunes a las 9:00 vía cron y envía el resultado por Telegram.
- `cron_weekly.sh` — script bash general del cron semanal: lanza cartelera filtrada y conciertos favoritos.
- `concerts_cron.py` — script independiente que consulta la API de Wegow, filtra los conciertos de la semana por artistas favoritos del perfil y envía el resultado por Telegram.

**Configuración y perfil**
- `config.py` — configuración centralizada: tokens de Telegram, URL y modelo de Ollama, rutas de caché y headers HTTP.
- `user_profile.json` — perfil del usuario con notas mínimas por género, directores favoritos y artistas favoritos para el filtrado.

**Frontends**
- `telegram_bot.py` — bot de Telegram con comandos `/pelicula`, `/cartelera`, `/conciertos` y texto libre; delega todas las consultas al agente.
- `web_app.py` — aplicación web Flask con buscador de películas, cartelera filtrada y API REST en `/api/q`.
- `templates/` — plantillas HTML de la interfaz web.
- `alexa_skill_def/` — carpeta con la lambda de Alexa (`lambda_function.py`), el modelo de interacción JSON con los intents y utterances, el módulo de caché con soporte opcional a DynamoDB, y el script de despliegue.

**Calendarios**
- `agenda_conciertos.ics` — calendario en formato iCalendar (RFC 5545) con los conciertos de la semana, importable en Google Calendar, Outlook o Apple Calendar.
- `agenda_cartelera.ics` — calendario equivalente con las películas en cartelera filtradas por el perfil del usuario.

**Workflows externos**
- `n8n_guardrail_workflow.json` — workflow de N8N con validación de entrada y salida del LLM: bloquea prompts maliciosos y filtra respuestas inapropiadas antes de devolverlas al usuario.
- `comfyui_acestep_workflow.json` — workflow de ComfyUI con el modelo Ace Step para generación de canciones a partir de tags de estilo y letra.

**Documentación y entorno**
- `README.md` — documentación completa del proyecto: arquitectura, instrucciones de instalación, ejemplos de uso y descripción de cada componente.
- `requirements.txt` — dependencias Python del proyecto.